# ML Zoomcamp 2026 - Homework 2: Regression

Решение домашнего задания по линейной регрессии на данных об экономичности автомобилей.

## Загрузка и подготовка данных

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "car_fuel_efficiency_2026.csv"
)

features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
]
target = "fuel_efficiency_mpg"

df = pd.read_csv(url)[features + [target]]
df.head()

## EDA: распределение целевой переменной

Гистограмма позволяет посмотреть, есть ли у `fuel_efficiency_mpg` длинный хвост. Выраженного длинного хвоста здесь нет.

In [ ]:
df[target].hist(bins=50)
plt.xlabel("Fuel efficiency, MPG")
plt.title("Distribution of fuel efficiency")
plt.show()

## Question 1: пропущенные значения

`isna()` отмечает пропуски, а `sum()` считает их в каждом столбце.

In [ ]:
missing_values = df.isna().sum()
display(missing_values)

missing_column = missing_values[missing_values > 0].index[0]
print("Answer Q1:", missing_column)

**Ответ Q1:** `horsepower`

## Question 2: медиана horsepower

Медиана является центральным значением: половина наблюдений находится ниже неё, половина выше.

In [ ]:
median_horsepower = df["horsepower"].median()
print("Median horsepower:", median_horsepower)
print("Answer Q2:", round(median_horsepower))

**Ответ Q2:** `254`

## Вспомогательные функции для вопросов 3-6

`split_data` перемешивает строки и делит данные в пропорции 60% / 20% / 20%. `train_linear_regression` вычисляет коэффициенты линейной регрессии. `rmse` измеряет среднюю величину ошибки модели.

In [ ]:
def split_data(dataframe, seed):
    n = len(dataframe)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    idx = np.arange(n)
    np.random.RandomState(seed).shuffle(idx)

    df_train = dataframe.iloc[idx[:n_train]]
    df_val = dataframe.iloc[idx[n_train:n_train + n_val]]
    df_test = dataframe.iloc[idx[n_train + n_val:]]

    return df_train, df_val, df_test


def train_linear_regression(X, y, r=0.0):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T @ X

    regularization = r * np.eye(XTX.shape[0])
    regularization[0, 0] = 0.0

    w = np.linalg.solve(XTX + regularization, X.T @ y)
    return w[0], w[1:]


def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))


def model_rmse(df_train, df_check, fill_value, r=0.0):
    X_train = df_train[features].fillna(fill_value).values
    y_train = df_train[target].values
    X_check = df_check[features].fillna(fill_value).values
    y_check = df_check[target].values

    w0, w = train_linear_regression(X_train, y_train, r=r)
    y_pred = w0 + X_check @ w

    return rmse(y_check, y_pred)

## Question 3: заполнение пропусков

Сравниваем заполнение пропусков нулём и средним значением `horsepower`. Среднее вычисляем только по обучающей части, чтобы не использовать информацию из validation. Меньший RMSE означает лучший результат.

In [ ]:
df_train, df_val, df_test = split_data(df, seed=42)

mean_horsepower = df_train["horsepower"].mean()
rmse_zero = model_rmse(df_train, df_val, fill_value=0.0)
rmse_mean = model_rmse(df_train, df_val, fill_value=mean_horsepower)

comparison_q3 = pd.DataFrame({
    "method": ["Fill with 0", "Fill with mean"],
    "RMSE": [round(rmse_zero, 3), round(rmse_mean, 3)],
})
display(comparison_q3)
print("Answer Q3: With mean")

**Ответ Q3:** `With mean`, потому что `2.202` меньше `2.205`.

## Question 4: выбор регуляризации

Проверяем все указанные значения `r`. Если после округления лучший результат одинаков для нескольких вариантов, выбираем самое маленькое `r`.

In [ ]:
regularization_values = [0, 0.01, 0.1, 1, 5, 10, 100]
scores_q4 = []

for r in regularization_values:
    score = model_rmse(df_train, df_val, fill_value=0.0, r=r)
    scores_q4.append(score)

results_q4 = pd.DataFrame({
    "r": regularization_values,
    "RMSE": scores_q4,
    "RMSE rounded": [round(score, 4) for score in scores_q4],
})
display(results_q4)

best_r = min(
    regularization_values,
    key=lambda value: (round(scores_q4[regularization_values.index(value)], 4), value),
)
print("Answer Q4:", best_r)

**Ответ Q4:** `0`

## Question 5: влияние seed

Для каждого `seed` данные перемешиваются немного иначе. Получаем десять значений RMSE и вычисляем их стандартное отклонение.

In [ ]:
scores_q5 = []

for seed in range(10):
    seed_train, seed_val, _ = split_data(df, seed=seed)
    score = model_rmse(seed_train, seed_val, fill_value=0.0)
    scores_q5.append(score)

display(pd.DataFrame({
    "seed": range(10),
    "RMSE": scores_q5,
}))

std = np.std(scores_q5)
print("Standard deviation:", std)
print("Answer Q5:", round(std, 3))

**Ответ Q5:** `0.029`

## Question 6: финальная модель

Используем `seed=9`, объединяем train и validation, заполняем пропуски нулями и обучаем модель с `r=0.001`. Затем измеряем RMSE на test.

In [ ]:
df_train_9, df_val_9, df_test_9 = split_data(df, seed=9)
df_full_train = pd.concat([df_train_9, df_val_9])

test_rmse = model_rmse(
    df_full_train,
    df_test_9,
    fill_value=0.0,
    r=0.001,
)

print("Test RMSE:", test_rmse)
print("Answer Q6:", round(test_rmse, 3))

**Ответ Q6:** `2.236`

## Итоговые ответы

1. `horsepower`
2. `254`
3. `With mean`
4. `0`
5. `0.029`
6. `2.236`